In [ ]:
# Simulation Study - Sensitivity of PE to heteroscedastic and Gaussian processes

In [ ]:
# IC Data Set


import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from patsy import dmatrix
from scipy.interpolate import make_interp_spline 

# --- 1. Simulate the Dataset of 1000 observations
np.random.seed(123) # Set seed for reproducibility of initial data generation

X = np.linspace(1, 20, 1000)
y_true_sin_segment = (np.sin(np.pi * X / 5) + 0.2 * np.cos(4 * np.pi * X / 5))
y_true_linear_segment = -1 + X / 10

y_true = np.zeros_like(X)
mask_sin = X <= 9.6
mask_linear = X > 9.6
y_true[mask_sin] = y_true_sin_segment[mask_sin]
y_true[mask_linear] = y_true_linear_segment[mask_linear]

noise_sd = 0.1
# Capture the generated noise
generated_noise = np.random.normal(loc=0, scale=noise_sd, size=len(y_true))
y = y_true + generated_noise
dat = pd.DataFrame({'x': X, 'y': y}) # Create a pandas DataFrame similar to R's

# --- FIRST PLOT: Standalone Generated Data ---
plt.figure(figsize=(10, 6))
plt.scatter(dat['x'], dat['y'], label='Generated Noisy Data', alpha=0.6, s=15, color='gray')
plt.plot(X, y_true, color='green', linestyle='--', linewidth=2, label='True Underlying Function')
plt.axvline(x=9.6, color='purple', linestyle=':', linewidth=1.5, label='Known Transition Point')

plt.title('Generated Data')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.legend()
plt.grid(True)
plt.show()

print(f"Generated {len(X)} original data points.")

# --- 2. Define a function to fit B-spline and predict on grid ---
def get_knots_from_df(data_x, df, degree):
    num_internal_knots = df - degree - 1
    if num_internal_knots < 0:
        return [] # No internal knots needed if df is too small
    # Calculate quantiles for internal knots
    knots = np.percentile(data_x, np.linspace(0, 100, num_internal_knots + 2)[1:-1])
    return knots

# Define spline parameters
k = 3 # Cubic spline degree
df_r_equivalent = 8 
# Calculate internal knots based on R's df logic
internal_knots_calculated = get_knots_from_df(dat['x'], df_r_equivalent, k)

# Function to fit B-spline and predict on grid
def fit_bspline_python(data_df, grid_x, degree, knots):
    # Ensure data_df has 'x' and 'y' columns
    # Create the B-spline basis matrix for the fitting data
    X_bs_fit = dmatrix(f"bs(x, knots={list(knots)}, degree={degree}, include_intercept=False)", {"x": data_df['x']})

    try:
        # Fit a linear model (Ordinary Least Squares)
        coefficients = np.linalg.lstsq(X_bs_fit, data_df['y'], rcond=1e-6)[0]

        # Create the B-spline basis matrix for the prediction grid
        X_bs_grid = dmatrix(f"bs(x, knots={list(knots)}, degree={degree}, include_intercept=False)", {"x": grid_x})

        # Predict y values on the grid
        yhat = X_bs_grid @ coefficients
    except np.linalg.LinAlgError as e:
        print(f"Warning: np.linalg.lstsq failed during spline fitting with error: {e}")
        yhat = np.full(len(grid_x), np.nan)
    return yhat

# --- 3. Fit to original data, extract residuals ---
grid = np.linspace(min(dat['x']), max(dat['x']), 1000)

# Fit to original data
fitted_spline_curve_orig = fit_bspline_python(dat, dat['x'], k, internal_knots_calculated)

# Check if the initial fit resulted in NaNs (due to LinAlgError)
if np.all(np.isnan(fitted_spline_curve_orig)):
    print("FATAL ERROR: Initial spline fit returned all NaNs. Cannot proceed with bootstrapping.")
    # Raise an error to stop execution if the fundamental fit failed
    raise ValueError("Initial spline fit failed, resulting in all NaNs.")

dat['yhat'] = fitted_spline_curve_orig
dat['resid'] = dat['y'] - dat['yhat']

# Check if residuals are all NaNs (should not happen if yhat is not all NaNs and y is valid)
if np.all(np.isnan(dat['resid'])):
    print("FATAL ERROR: Residuals are all NaNs after initial spline fit. Cannot proceed.")
    raise ValueError("Residuals are invalid, cannot proceed with bootstrapping.")

# --- 4. Bootstrap loop (Nonparametric Residual Bootstrap) ---
B = 200 # Number of bootstrap replicates
boot_preds_list = [] # To store predicted y-values from each bootstrap fit
simulated_y_datasets = [] # To store the simulated y datasets (before refitting)

# Set a random seed for reproducibility of the residual bootstrapping
np.random.seed(123)

for i in range(B):
    # Resample residuals with replacement
    resampled_resid = np.random.choice(dat['resid'], size=len(dat), replace=True)

    # Create new 'b' (bootstrapped y) by adding resampled residuals to original fitted values
    dat['b'] = dat['yhat'] + resampled_resid
    simulated_y_datasets.append(dat['b'].copy()) # Store this simulated y dataset

    # Fit spline to the new 'b' data (using original X values)
    # Create a temporary DataFrame for the bootstrap sample
    bootstrap_data_df = pd.DataFrame({'x': dat['x'], 'y': dat['b']})

    # Predict on the common grid
    preds_b_yhat = fit_bspline_python(bootstrap_data_df, grid, k, internal_knots_calculated)
    boot_preds_list.append(preds_b_yhat)

boot_preds_array = np.array(boot_preds_list) # Convert list to numpy array
simulated_y_datasets_array = np.array(simulated_y_datasets) # NEW: Convert list to numpy array


# --- Calculate statistics for plotting (mean and CI) ---
mean_boot_fit = np.mean(boot_preds_array, axis=0)
lower_boot_ci = np.percentile(boot_preds_array, 2.5, axis=0)
upper_boot_ci = np.percentile(boot_preds_array, 97.5, axis=0)

# --- CSV Export of Bootstrap Fits ---
# Transpose the boot_preds_array so that each fit is a column
bootstrap_data_transposed = boot_preds_array.T

# Create column names for the DataFrame
bootstrap_column_names = [f"Simulation_{i+1}" for i in range(B)]

# Create a Pandas DataFrame
df_bootstrap_fits = pd.DataFrame(bootstrap_data_transposed, columns=bootstrap_column_names)

# Add the 'Time Step/Simulation' column
df_bootstrap_fits.insert(0, 'Time Step/Simulation', np.arange(1, len(grid) + 1))

# Define the CSV file name
csv_bootstrap_file_name = 'Spline_Simulations.csv'

# Save the DataFrame to a CSV file
df_bootstrap_fits.to_csv(csv_bootstrap_file_name, index=False)

print(f"\nSaved {B} bootstrap spline fits to '{csv_bootstrap_file_name}'")
print(f"CSV file shape: {df_bootstrap_fits.shape} (rows: time steps, columns: bootstrap fits)")


# --- SECOND PLOT
plt.figure(figsize=(12, 7))

# Plot original noisy data 
plt.scatter(dat['x'], dat['y'], color='black', alpha=0.3, s=10, label='Original Noisy Data')

# Plot the true underlying function 
plt.plot(X, y_true, color='green', linestyle='--', linewidth=2, label='True Underlying Function', alpha=0.8)

# Plot the single fitted B-spline (from R: blue, size=1; Python: red) - using blue for R's consistency
plt.plot(dat['x'], fitted_spline_curve_orig, color='blue', linewidth=4, label='Single Fitted B-spline')

plt.title('IC Data Set')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.legend(fontsize=20)
plt.grid(True)
plt.show()


In [ ]:
# OOC Data Set 1


# --- Import necessary libraries ---
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from patsy import dmatrix

# --- 1. Simulate the Dataset with Heteroscedastic and High-Frequency Noise ---
np.random.seed(123) 

X = np.linspace(0.1, 10, 1000)
y_true = np.zeros_like(X)
mask_segment_1 = X <= 4
mask_segment_2 = X > 4

# Use new, complex equations for the true function
y_true[mask_segment_1] = 1.2 * np.cos(np.pi * X[mask_segment_1]) + 0.5 * X[mask_segment_1] + 1.2 + 0.15 * np.sin(10 * np.pi * X[mask_segment_1])
y_true[mask_segment_2] = 0.2 * X[mask_segment_2] + 0.8 * np.sin(np.pi * (X[mask_segment_2] - 4) / 2) + 3.6 + 0.15 * np.sin(10 * np.pi * X[mask_segment_2])

# Generate heteroscedastic noise
noise_sd_new = 0.08 + 0.04 * X
generated_noise = np.random.normal(loc=0, scale=noise_sd_new, size=len(y_true))
y = y_true + generated_noise
dat = pd.DataFrame({'x': X, 'y': y}) 

# --- Plot 1: Standalone Generated Data ---
plt.figure(figsize=(10, 6))
plt.scatter(dat['x'], dat['y'], label='Generated Noisy Data', alpha=0.6, s=15, color='gray')
plt.plot(X, y_true, color='green', linestyle='--', linewidth=2, label='True Underlying Function')
plt.axvline(x=4, color='purple', linestyle=':', linewidth=1.5, label='Known Transition Point')
plt.title('Generated Data with Heteroscedastic and High-Frequency Noise')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.legend()
plt.grid(True)
plt.show()

print(f"Generated {len(X)} original data points.")

# --- 2. Spline fitting and bootstrapping ---
k = 3
df_r_equivalent = 15
def get_knots_from_df(data_x, df, degree):
    num_internal_knots = df - degree - 1
    if num_internal_knots < 0: return []
    knots = np.percentile(data_x, np.linspace(0, 100, num_internal_knots + 2)[1:-1])
    return knots

internal_knots_calculated = get_knots_from_df(dat['x'], df_r_equivalent, k)
def fit_bspline_python(data_df, grid_x, degree, knots):
    X_bs_fit = dmatrix(f"bs(x, knots={list(knots)}, degree={degree}, include_intercept=False)", {"x": data_df['x']})
    try:
        coefficients = np.linalg.lstsq(X_bs_fit, data_df['y'], rcond=1e-6)[0]
        X_bs_grid = dmatrix(f"bs(x, knots={list(knots)}, degree={degree}, include_intercept=False)", {"x": grid_x})
        yhat = X_bs_grid @ coefficients
    except np.linalg.LinAlgError as e:
        yhat = np.full(len(grid_x), np.nan)
    return yhat

grid = np.linspace(min(dat['x']), max(dat['x']), 1000)
fitted_spline_curve_orig = fit_bspline_python(dat, dat['x'], k, internal_knots_calculated)
if np.all(np.isnan(fitted_spline_curve_orig)):
    raise ValueError("Initial spline fit failed, resulting in all NaNs.")

dat['yhat'] = fitted_spline_curve_orig
dat['resid'] = dat['y'] - dat['yhat']
if np.all(np.isnan(dat['resid'])):
    raise ValueError("Residuals are invalid, cannot proceed with bootstrapping.")

B = 200
boot_preds_list = []
np.random.seed(123) 

for i in range(B):
    resampled_resid = np.random.choice(dat['resid'], size=len(dat), replace=True)
    dat['b'] = dat['yhat'] + resampled_resid
    bootstrap_data_df = pd.DataFrame({'x': dat['x'], 'y': dat['b']})
    preds_b_yhat = fit_bspline_python(bootstrap_data_df, grid, k, internal_knots_calculated)
    boot_preds_list.append(preds_b_yhat)
boot_preds_array = np.array(boot_preds_list)

mean_boot_fit = np.mean(boot_preds_array, axis=0)
lower_boot_ci = np.percentile(boot_preds_array, 2.5, axis=0)
upper_boot_ci = np.percentile(boot_preds_array, 97.5, axis=0)

# --- CSV Export of Bootstrap Fits ---
# Transpose the boot_preds_array so that each fit is a column
bootstrap_data_transposed = boot_preds_array.T
bootstrap_column_names = [f"Simulation_{i+1}" for i in range(B)]
df_bootstrap_fits = pd.DataFrame(bootstrap_data_transposed, columns=bootstrap_column_names)
df_bootstrap_fits.insert(0, 'Time Step/Simulation', np.arange(1, len(grid) + 1))
csv_bootstrap_file_name = 'Spline_Simulations_OOC_New_1.csv'
df_bootstrap_fits.to_csv(csv_bootstrap_file_name, index=False)
print(f"\nSaved {B} bootstrap spline fits to '{csv_bootstrap_file_name}'")
print(f"CSV file shape: {df_bootstrap_fits.shape} (rows: time steps, columns: bootstrap fits)")

# --- Plot 2: Combined Visualization ---
plt.figure(figsize=(12, 7))
plt.scatter(dat['x'], dat['y'], color='black', alpha=0.3, s=10, label='Original Noisy Data')
plt.plot(X, y_true, color='green', linestyle='--', linewidth=2, label='True Underlying Function')
plt.plot(dat['x'], fitted_spline_curve_orig, color='blue', linewidth=4, label='Single Fitted B-spline')

plt.title('OOC Data Set 1')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.legend(fontsize=20)
plt.grid(True)
plt.show

In [ ]:
# OOC Data Set 2


# --- Import necessary libraries ---
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from patsy import dmatrix
from fbm import fbm, fgn, MBM # Library for Fractional Brownian Motion
from scipy.interpolate import make_interp_spline 

# --- 1. Simulate the Dataset
np.random.seed(123) 

X = np.linspace(1, 20, 1000)
y_true = np.zeros_like(X)
mask_sin = X <= 9.6
mask_fbm = X > 9.6

# First segment of the true function (Original sine/cosine - same as the first part of the IC Data Set)
y_true_sin_segment = (np.sin(np.pi * X[mask_sin] / 5) + 0.2 * np.cos(4 * np.pi * X[mask_sin] / 5))
y_true[mask_sin] = y_true_sin_segment

# --- Generate the fBm component ---
hurst_param = 0.1
# The fBm path
fbm_path = fbm(n=len(X[mask_fbm]), hurst=hurst_param, length=len(X[mask_fbm]), method='daviesharte')

start_y_fbm = y_true[mask_sin][-1]
fbm_scaled = fbm_path[:-1] * 0.5 + start_y_fbm - fbm_path[0]
y_true[mask_fbm] = fbm_scaled + 0.05 * (X[mask_fbm] - 9.6) 

noise_sd = 0.1
generated_noise = np.random.normal(loc=0, scale=noise_sd, size=len(y_true))
y = y_true + generated_noise
dat = pd.DataFrame({'x': X, 'y': y})

# --- Plot 1: Standalone Generated Data ---
plt.figure(figsize=(10, 6))
plt.scatter(dat['x'], dat['y'], label='Generated Noisy Data', alpha=0.6, s=15, color='gray')
plt.plot(X, y_true, color='green', linestyle='--', linewidth=2, label='True Underlying Function')
plt.axvline(x=9.6, color='purple', linestyle=':', linewidth=1.5, label='Known Transition Point')
plt.title('Generated Data with fBm Component (H=0.1)')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.legend()
plt.grid(True)
plt.show()

print(f"Generated {len(X)} original data points.")

# --- 2. Spline fitting and bootstrapping ---
k = 3
df_r_equivalent = 8
def get_knots_from_df(data_x, df, degree):
    num_internal_knots = df - degree - 1
    if num_internal_knots < 0: return []
    knots = np.percentile(data_x, np.linspace(0, 100, num_internal_knots + 2)[1:-1])
    return knots

internal_knots_calculated = get_knots_from_df(dat['x'], df_r_equivalent, k)
def fit_bspline_python(data_df, grid_x, degree, knots):
    X_bs_fit = dmatrix(f"bs(x, knots={list(knots)}, degree={degree}, include_intercept=False)", {"x": data_df['x']})
    try:
        coefficients = np.linalg.lstsq(X_bs_fit, data_df['y'], rcond=1e-6)[0]
        X_bs_grid = dmatrix(f"bs(x, knots={list(knots)}, degree={degree}, include_intercept=False)", {"x": grid_x})
        yhat = X_bs_grid @ coefficients
    except np.linalg.LinAlgError as e:
        yhat = np.full(len(grid_x), np.nan)
    return yhat

grid = np.linspace(min(dat['x']), max(dat['x']), 1000)
fitted_spline_curve_orig = fit_bspline_python(dat, dat['x'], k, internal_knots_calculated)
if np.all(np.isnan(fitted_spline_curve_orig)):
    raise ValueError("Initial spline fit failed, resulting in all NaNs.")

dat['yhat'] = fitted_spline_curve_orig
dat['resid'] = dat['y'] - dat['yhat']
if np.all(np.isnan(dat['resid'])):
    raise ValueError("Residuals are invalid, cannot proceed with bootstrapping.")

B = 200
boot_preds_list = []
np.random.seed(123) 

for i in range(B):
    resampled_resid = np.random.choice(dat['resid'], size=len(dat), replace=True)
    dat['b'] = dat['yhat'] + resampled_resid
    bootstrap_data_df = pd.DataFrame({'x': dat['x'], 'y': dat['b']})
    preds_b_yhat = fit_bspline_python(bootstrap_data_df, grid, k, internal_knots_calculated)
    boot_preds_list.append(preds_b_yhat)
boot_preds_array = np.array(boot_preds_list)

mean_boot_fit = np.mean(boot_preds_array, axis=0)
lower_boot_ci = np.percentile(boot_preds_array, 2.5, axis=0)
upper_boot_ci = np.percentile(boot_preds_array, 97.5, axis=0)

# --- CSV Export of Bootstrap Fits ---
bootstrap_data_transposed = boot_preds_array.T
bootstrap_column_names = [f"Simulation_{i+1}" for i in range(B)]
df_bootstrap_fits = pd.DataFrame(bootstrap_data_transposed, columns=bootstrap_column_names)
df_bootstrap_fits.insert(0, 'Time Step/Simulation', np.arange(1, len(grid) + 1))
csv_bootstrap_file_name = 'Spline_Simulations_OOC_New_2.csv'
df_bootstrap_fits.to_csv(csv_bootstrap_file_name, index=False)
print(f"\nSaved {B} bootstrap spline fits to '{csv_bootstrap_file_name}'")
print(f"CSV file shape: {df_bootstrap_fits.shape} (rows: time steps, columns: bootstrap fits)")

# --- Plot 2: Combined Visualization ---
plt.figure(figsize=(12, 7))
plt.scatter(dat['x'], dat['y'], color='black', alpha=0.3, s=10, label='Original Noisy Data')
plt.plot(X, y_true, color='green', linestyle='--', linewidth=2, label='True Underlying Function')
plt.plot(dat['x'], fitted_spline_curve_orig, color='blue', linewidth=4, label='Single Fitted B-spline')

plt.title('OOC Data Set 2')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.legend(fontsize=20)
plt.grid(True)
plt.show

In [ ]:
# --- Import necessary libraries ---
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from patsy import dmatrix
from scipy.interpolate import make_interp_spline 

# --- 1. Simulate Dataset 
np.random.seed(123) # Set seed for reproducibility of initial data generation

X = np.linspace(1, 20, 1000)
y_true = np.zeros_like(X)
mask_sin = X <= 9.6
mask_gaussian = X > 9.6

# First segment of the true function (Original sine/cosine - same as the first part of the IC Data Set)
y_true_sin_segment = (np.sin(np.pi * X[mask_sin] / 5) + 0.2 * np.cos(4 * np.pi * X[mask_sin] / 5))
y_true[mask_sin] = y_true_sin_segment

# --- Generate the strict Gaussian noise and replace the linear segment ---
start_y_gaussian = y_true[mask_sin][-1]
# Now, generate homoscedastic Gaussian noise directly for the second segment
y_true[mask_gaussian] = np.random.normal(loc=start_y_gaussian, scale=0.1, size=len(X[mask_gaussian]))

# Use the original constant noise level
noise_sd = 0.1
generated_noise = np.random.normal(loc=0, scale=noise_sd, size=len(y_true))
y = y_true + generated_noise
dat = pd.DataFrame({'x': X, 'y': y})

# --- FIRST PLOT: Standalone Generated Data ---
plt.figure(figsize=(10, 6))
plt.scatter(dat['x'], dat['y'], label='Generated Noisy Data', alpha=0.6, s=15, color='gray')
plt.plot(X, y_true, color='green', linestyle='--', linewidth=2, label='True Underlying Function')
plt.axvline(x=9.6, color='purple', linestyle=':', linewidth=1.5, label='Known Transition Point')
plt.title('Generated Data with Strict Gaussian Noise Segment')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.legend()
plt.grid(True)
plt.show()

print(f"Generated {len(X)} original data points.")

# --- 2. Spline fitting and bootstrapping ---
k = 3 # Cubic spline degree
df_r_equivalent = 8 
def get_knots_from_df(data_x, df, degree):
    num_internal_knots = df - degree - 1
    if num_internal_knots < 0: return []
    knots = np.percentile(data_x, np.linspace(0, 100, num_internal_knots + 2)[1:-1])
    return knots

internal_knots_calculated = get_knots_from_df(dat['x'], df_r_equivalent, k)

def fit_bspline_python(data_df, grid_x, degree, knots):
    X_bs_fit = dmatrix(f"bs(x, knots={list(knots)}, degree={degree}, include_intercept=False)", {"x": data_df['x']})
    try:
        coefficients = np.linalg.lstsq(X_bs_fit, data_df['y'], rcond=1e-6)[0]
        X_bs_grid = dmatrix(f"bs(x, knots={list(knots)}, degree={degree}, include_intercept=False)", {"x": grid_x})
        yhat = X_bs_grid @ coefficients
    except np.linalg.LinAlgError as e:
        yhat = np.full(len(grid_x), np.nan)
    return yhat

grid = np.linspace(min(dat['x']), max(dat['x']), 1000)
fitted_spline_curve_orig = fit_bspline_python(dat, dat['x'], k, internal_knots_calculated)
if np.all(np.isnan(fitted_spline_curve_orig)):
    raise ValueError("Initial spline fit failed, resulting in all NaNs.")

dat['yhat'] = fitted_spline_curve_orig
dat['resid'] = dat['y'] - dat['yhat']
if np.all(np.isnan(dat['resid'])):
    raise ValueError("Residuals are invalid, cannot proceed with bootstrapping.")

B = 200
boot_preds_list = []
np.random.seed(123) 

for i in range(B):
    resampled_resid = np.random.choice(dat['resid'], size=len(dat), replace=True)
    dat['b'] = dat['yhat'] + resampled_resid
    bootstrap_data_df = pd.DataFrame({'x': dat['x'], 'y': dat['b']})
    preds_b_yhat = fit_bspline_python(bootstrap_data_df, grid, k, internal_knots_calculated)
    boot_preds_list.append(preds_b_yhat)
boot_preds_array = np.array(boot_preds_list)

mean_boot_fit = np.mean(boot_preds_array, axis=0)
lower_boot_ci = np.percentile(boot_preds_array, 2.5, axis=0)
upper_boot_ci = np.percentile(boot_preds_array, 97.5, axis=0)

# --- CSV Export of Bootstrap Fits ---
bootstrap_data_transposed = boot_preds_array.T
bootstrap_column_names = [f"Simulation_{i+1}" for i in range(B)]
df_bootstrap_fits = pd.DataFrame(bootstrap_data_transposed, columns=bootstrap_column_names)
df_bootstrap_fits.insert(0, 'Time Step/Simulation', np.arange(1, len(grid) + 1))
csv_bootstrap_file_name = 'Spline_Simulations_OOC_New_3.csv'
df_bootstrap_fits.to_csv(csv_bootstrap_file_name, index=False)
print(f"\nSaved {B} bootstrap spline fits to '{csv_bootstrap_file_name}'")
print(f"CSV file shape: {df_bootstrap_fits.shape} (rows: time steps, columns: bootstrap fits)")

# --- SECOND PLOT: Combined Visualization ---
plt.figure(figsize=(12, 7))
plt.scatter(dat['x'], dat['y'], color='black', alpha=0.3, s=10, label='Original Noisy Data')
plt.plot(X, y_true, color='green', linestyle='--', linewidth=2, label='True Underlying Function')
plt.plot(dat['x'], fitted_spline_curve_orig, color='blue', linewidth=4, label='Single Fitted B-spline')

plt.title('OOC Data Set 3')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.legend(fontsize=20)
plt.grid(True)
plt.show